# Notebook 5 — Part 5: Design Decisions and Analysis

Project: **SVG-GPT: Scaling Transformer Language Models for Vector Graphics**

This notebook addresses **Part 5: Design Decisions and Analysis**.

Unlike the previous notebooks, this notebook didn't train a model. It consolidated the completed experimental results from Notebooks 1–4 and documented the design choices, tradeoffs, results, limitations, and conclusions from the project.

The analysis in this notebook answered the seven required Part 5 questions:

1. Tokenization strategy
2. Architecture choices
3. Training decisions
4. Scaling insights
5. Learning-rate scaling insights
6. SVG-specific patterns
7. Challenges encountered and future work

It read result files from Google Drive, created final tables/plots where useful, and wrote a report-ready Part 5 analysis document.

In [1]:
import os
import re
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPyImage

print("Imports complete.")

Imports complete.


## 1. Mount Google Drive and Locate Project Outputs

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_ROOT = Path("/content/drive/MyDrive/svg_gpt_scaling")

NOTEBOOK1_DIR = PROJECT_ROOT / "notebook1_preprocessing"
NOTEBOOK2_DIR = PROJECT_ROOT / "notebook2_standard_scaling"
NOTEBOOK3_DIR = PROJECT_ROOT / "notebook3_mup_scaling"
NOTEBOOK4_DIR = PROJECT_ROOT / "notebook4_generation_eval"

FINAL_DIR = PROJECT_ROOT / "final_report_assets"
PART5_DIR = FINAL_DIR / "part5_design_analysis"
TABLES_DIR = PART5_DIR / "tables"
FIGURES_DIR = PART5_DIR / "figures"
REPORT_TEXT_DIR = PART5_DIR / "report_text"
REPO_FILES_DIR = FINAL_DIR / "repo_files"

for d in [FINAL_DIR, PART5_DIR, TABLES_DIR, FIGURES_DIR, REPORT_TEXT_DIR, REPO_FILES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Part 5 output folder:", PART5_DIR)

Project root: /content/drive/MyDrive/svg_gpt_scaling
Part 5 output folder: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis


## 2. Helper Functions

In [4]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        print("Missing JSON:", path)
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_csv(path):
    path = Path(path)
    if not path.exists():
        print("Missing CSV:", path)
        return None
    return pd.read_csv(path)


def save_markdown(text, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    print("Saved:", path)


def safe(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default


def copy_if_exists(src, dst):
    src = Path(src)
    dst = Path(dst)
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True
    print("Missing:", src)
    return False


print("Helper functions ready.")

Helper functions ready.


## 3. Load Actual Results from Notebooks 1–4

The following cells loaded the actual saved outputs from the completed notebooks. These values were used to support the Part 5 analysis rather than relying on guesses.

In [5]:
# Notebook 1 outputs
preprocess_config = read_json(NOTEBOOK1_DIR / "preprocess_config_full.json")
clean_stats = read_csv(NOTEBOOK1_DIR / "stats" / "clean_stats_full.csv")
split_token_summary = read_csv(NOTEBOOK1_DIR / "stats" / "split_token_summary_full.csv")
token_lengths = read_csv(NOTEBOOK1_DIR / "stats" / "token_lengths_full.csv")

# Notebook 2 outputs
standard_scaling = read_csv(NOTEBOOK2_DIR / "results" / "standard_scaling_results_real.csv")
standard_arch = read_csv(NOTEBOOK2_DIR / "results" / "standard_model_architectures.csv")
standard_lr_sweep = read_csv(NOTEBOOK2_DIR / "results" / "lr_sweep_tiny_standard_real.csv")
standard_fit = read_json(NOTEBOOK2_DIR / "results" / "standard_scaling_law_fit_real.json")

# Notebook 3 outputs
mup_scaling = read_csv(NOTEBOOK3_DIR / "results" / "mup_scaling_results_real.csv")
mup_arch = read_csv(NOTEBOOK3_DIR / "results" / "mup_model_architectures.csv")
mup_lr_sweep = read_csv(NOTEBOOK3_DIR / "results" / "lr_sweep_tiny_mup_real.csv")
mup_fit = read_json(NOTEBOOK3_DIR / "results" / "mup_scaling_law_fit_real.json")
comparison_table = read_csv(NOTEBOOK3_DIR / "results" / "standard_vs_mup_comparison_table_real.csv")
extrapolation = read_json(NOTEBOOK3_DIR / "results" / "scaling_extrapolation_10x_real.json")

# Notebook 4 outputs
part4_metrics = read_json(NOTEBOOK4_DIR / "metrics" / "part4_quantitative_metrics.json")
initial_best_metrics = read_json(NOTEBOOK4_DIR / "metrics" / "initial_best_model_metrics.json")
final_best_metrics = read_json(NOTEBOOK4_DIR / "metrics" / "final_best_model_metrics.json")
candidate_metrics = read_csv(NOTEBOOK4_DIR / "metrics" / "all_generated_candidate_metrics.csv")
curated_samples = read_csv(NOTEBOOK4_DIR / "metrics" / "curated_report_samples_metrics.csv")
temperature_summary = read_csv(NOTEBOOK4_DIR / "metrics" / "temperature_summary_unconditional.csv")
prefix_completion_table = read_csv(NOTEBOOK4_DIR / "report_assets" / "prefix_completion_report_table.csv")

print("Result files loaded.")

Result files loaded.


## 4. Evidence Checklist

In [6]:
expected_files = {
    "Part 1 preprocessing config": NOTEBOOK1_DIR / "preprocess_config_full.json",
    "Part 1 split token summary": NOTEBOOK1_DIR / "stats" / "split_token_summary_full.csv",
    "Part 2 standard scaling results": NOTEBOOK2_DIR / "results" / "standard_scaling_results_real.csv",
    "Part 2 standard LR sweep": NOTEBOOK2_DIR / "results" / "lr_sweep_tiny_standard_real.csv",
    "Part 2 standard scaling plot": NOTEBOOK2_DIR / "plots" / "standard_scaling_law_real.png",
    "Part 3 µP scaling results": NOTEBOOK3_DIR / "results" / "mup_scaling_results_real.csv",
    "Part 3 standard-vs-µP comparison": NOTEBOOK3_DIR / "plots" / "standard_vs_mup_scaling_real.png",
    "Part 3 extrapolation": NOTEBOOK3_DIR / "results" / "scaling_extrapolation_10x_real.json",
    "Part 4 generation metrics": NOTEBOOK4_DIR / "metrics" / "part4_quantitative_metrics.json",
    "Part 4 candidate metrics": NOTEBOOK4_DIR / "metrics" / "all_generated_candidate_metrics.csv",
    "Part 4 curated generated grid": NOTEBOOK4_DIR / "report_assets" / "unconditional_generated_grid_curated.png",
}

check_rows = []
for item, path in expected_files.items():
    check_rows.append({"item": item, "path": str(path), "exists": Path(path).exists()})

completion_check = pd.DataFrame(check_rows)
display(completion_check)
completion_check.to_csv(TABLES_DIR / "part5_evidence_checklist.csv", index=False)

,item,path,exists
0,Part 1 preprocessing config,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
1,Part 1 split token summary,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
2,Part 2 standard scaling results,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
3,Part 2 standard LR sweep,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
4,Part 2 standard scaling plot,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
5,Part 3 µP scaling results,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
6,Part 3 standard-vs-µP comparison,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
7,Part 3 extrapolation,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
8,Part 4 generation metrics,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True
9,Part 4 candidate metrics,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True


## 5. Key Numerical Results Used in the Analysis

In [7]:
if split_token_summary is not None:
    display(split_token_summary)
    split_token_summary.to_csv(TABLES_DIR / "part1_split_token_summary.csv", index=False)

if token_lengths is not None and len(token_lengths):
    token_length_summary = pd.DataFrame([{
        "num_examples_before_token_filter": int(len(token_lengths)),
        "mean_token_length": float(token_lengths["token_len"].mean()),
        "median_token_length": float(token_lengths["token_len"].median()),
        "min_token_length": int(token_lengths["token_len"].min()),
        "max_token_length": int(token_lengths["token_len"].max()),
        "p90_token_length": float(token_lengths["token_len"].quantile(0.90)),
        "p95_token_length": float(token_lengths["token_len"].quantile(0.95)),
        "p99_token_length": float(token_lengths["token_len"].quantile(0.99)),
    }])
else:
    token_length_summary = pd.DataFrame()

display(token_length_summary)
token_length_summary.to_csv(TABLES_DIR / "part1_token_length_summary.csv", index=False)

if standard_scaling is not None:
    standard_summary = standard_scaling[[
        "model_name", "params", "final_train_loss", "final_val_loss", "best_val_loss",
        "total_elapsed_sec", "tokens_per_sec", "gpu_memory_gb", "max_steps", "tokens_seen"
    ]].copy()
    standard_summary["params_millions"] = standard_summary["params"] / 1e6
    display(standard_summary)
    standard_summary.to_csv(TABLES_DIR / "part2_standard_scaling_summary.csv", index=False)

if mup_scaling is not None:
    mup_summary = mup_scaling[[
        "model_name", "params", "final_train_loss", "final_val_loss", "best_val_loss",
        "total_elapsed_sec", "tokens_per_sec", "gpu_memory_gb", "max_steps", "tokens_seen"
    ]].copy()
    mup_summary["params_millions"] = mup_summary["params"] / 1e6
    display(mup_summary)
    mup_summary.to_csv(TABLES_DIR / "part3_mup_scaling_summary.csv", index=False)

if part4_metrics is not None:
    generation_metrics_table = pd.DataFrame([part4_metrics])
    display(generation_metrics_table)
    generation_metrics_table.to_csv(TABLES_DIR / "part4_generation_metrics_summary.csv", index=False)

print("Saved key numerical tables.")

,split,files,tokens,bin_path,lengths_path,avg_tokens_per_file
0,train,255173,164389332,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,644.226983
1,val,2603,1657304,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,636.689973
2,test,2605,1655554,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,635.529367


,num_examples_before_token_filter,mean_token_length,median_token_length,min_token_length,max_token_length,p90_token_length,p95_token_length,p99_token_length
0,285731,874.111678,553.0,50,10648,1932.0,2627.0,4911.7


,model_name,params,final_train_loss,final_val_loss,best_val_loss,total_elapsed_sec,tokens_per_sec,gpu_memory_gb,max_steps,tokens_seen,params_millions
0,Tiny,1383168,0.890887,0.911614,0.876584,217.955544,754118.408678,0.562698,5016,164364288,1.383168
1,Small,3554304,0.786774,0.801394,0.792083,306.065544,537023.168015,0.779127,5016,164364288,3.554304
2,Medium,12417024,0.738939,0.773271,0.754734,308.336381,533068.097540,1.262346,5016,164364288,12.417024
3,Large,33884160,0.740568,0.767520,0.747741,492.315548,333859.632520,2.407393,5016,164364288,33.884160
4,XL,88594944,2.639022,2.684681,1.420349,838.323965,196062.971891,4.479008,5016,164364288,88.594944


,model_name,params,final_train_loss,final_val_loss,best_val_loss,total_elapsed_sec,tokens_per_sec,gpu_memory_gb,max_steps,tokens_seen,params_millions
0,Tiny,1907456,0.931006,0.952466,0.947877,280.839569,585260.434670,0.949877,5016,164364288,1.907456
1,Small,4340736,0.816816,0.839726,0.836315,398.822277,412124.140372,1.641756,5016,164364288,4.340736
2,Medium,13989888,0.786424,0.824152,0.818299,445.701535,368776.580312,2.133764,5016,164364288,13.989888
3,Large,35981312,0.782735,0.795803,0.784872,969.682701,169503.166151,4.323339,5016,164364288,35.981312
4,XL,91740672,0.811317,0.829749,0.822870,1734.061587,94785.726900,7.908979,5016,164364288,91.740672


,selected_model_source,params,additional_train_steps,final_test_loss,final_test_perplexity,final_val_loss,final_val_perplexity,num_unconditional_candidates,num_prefix_candidates,num_total_candidates,...,unconditional_xml_valid_rate,unconditional_render_rate,unconditional_structural_validity_rate,unconditional_report_ready_rate,prefix_xml_valid_rate,prefix_render_rate,prefix_structural_validity_rate,prefix_report_ready_rate,num_curated_unconditional_report_samples,num_curated_prefix_report_samples
0,standard_large,33884160,2500,0.715334,2.04487,0.714347,2.042852,80,80,160,...,0.475,0.475,0.475,0.3125,0.0875,0.0875,0.0875,0.0,12,0


Saved key numerical tables.


## 6. Part 5.1 — Tokenization Strategy

I used a **BPE tokenizer trained directly on the cleaned SVG corpus** rather than a natural-language tokenizer. This was appropriate because SVG code contains XML tags, numeric coordinates, path commands, attribute names, color values, and repeated syntax patterns that are very different from ordinary English. A domain-specific tokenizer allowed frequently occurring SVG fragments such as tags, attributes, and path substrings to become compact tokens instead of being split into inefficient character-level pieces.

I used a vocabulary size of **4096**, which was in the project-recommended range of 1K–8K. I chose 4096 as a middle-ground value: it was large enough to compress common SVG patterns, but small enough to keep the output softmax manageable for small models. A much smaller vocabulary would have increased sequence lengths and made the context window less effective. A much larger vocabulary would have increased embedding/output-layer parameters and could have been inefficient for the Tiny and Small models.

The preprocessing pipeline normalized SVG text before tokenization. I removed comments and metadata, normalized whitespace, rounded numeric precision, filtered very short and very long SVGs, validated XML, and deduplicated examples. This reduced vocabulary noise and made the token distribution more learnable.

The tokenizer produced a median SVG length of a few hundred tokens, while long examples could still exceed the context limit. For this reason, I filtered examples above **2048 tokens** and trained the Transformer using a context window of **512 tokens**. This was a practical compromise: it did not capture every long SVG completely, but it kept training feasible across all model sizes and allowed consistent scaling experiments.

In [8]:
section_text = '\n## 6. Part 5.1 — Tokenization Strategy\n\nI used a **BPE tokenizer trained directly on the cleaned SVG corpus** rather than a natural-language tokenizer. This was appropriate because SVG code contains XML tags, numeric coordinates, path commands, attribute names, color values, and repeated syntax patterns that are very different from ordinary English. A domain-specific tokenizer allowed frequently occurring SVG fragments such as tags, attributes, and path substrings to become compact tokens instead of being split into inefficient character-level pieces.\n\nI used a vocabulary size of **4096**, which was in the project-recommended range of 1K–8K. I chose 4096 as a middle-ground value: it was large enough to compress common SVG patterns, but small enough to keep the output softmax manageable for small models. A much smaller vocabulary would have increased sequence lengths and made the context window less effective. A much larger vocabulary would have increased embedding/output-layer parameters and could have been inefficient for the Tiny and Small models.\n\nThe preprocessing pipeline normalized SVG text before tokenization. I removed comments and metadata, normalized whitespace, rounded numeric precision, filtered very short and very long SVGs, validated XML, and deduplicated examples. This reduced vocabulary noise and made the token distribution more learnable.\n\nThe tokenizer produced a median SVG length of a few hundred tokens, while long examples could still exceed the context limit. For this reason, I filtered examples above **2048 tokens** and trained the Transformer using a context window of **512 tokens**. This was a practical compromise: it did not capture every long SVG completely, but it kept training feasible across all model sizes and allowed consistent scaling experiments.\n'
save_markdown(section_text, REPORT_TEXT_DIR / "part5_1_tokenization_strategy.md")
display(Markdown(section_text))

Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_1_tokenization_strategy.md



## 6. Part 5.1 — Tokenization Strategy

I used a **BPE tokenizer trained directly on the cleaned SVG corpus** rather than a natural-language tokenizer. This was appropriate because SVG code contains XML tags, numeric coordinates, path commands, attribute names, color values, and repeated syntax patterns that are very different from ordinary English. A domain-specific tokenizer allowed frequently occurring SVG fragments such as tags, attributes, and path substrings to become compact tokens instead of being split into inefficient character-level pieces.

I used a vocabulary size of **4096**, which was in the project-recommended range of 1K–8K. I chose 4096 as a middle-ground value: it was large enough to compress common SVG patterns, but small enough to keep the output softmax manageable for small models. A much smaller vocabulary would have increased sequence lengths and made the context window less effective. A much larger vocabulary would have increased embedding/output-layer parameters and could have been inefficient for the Tiny and Small models.

The preprocessing pipeline normalized SVG text before tokenization. I removed comments and metadata, normalized whitespace, rounded numeric precision, filtered very short and very long SVGs, validated XML, and deduplicated examples. This reduced vocabulary noise and made the token distribution more learnable.

The tokenizer produced a median SVG length of a few hundred tokens, while long examples could still exceed the context limit. For this reason, I filtered examples above **2048 tokens** and trained the Transformer using a context window of **512 tokens**. This was a practical compromise: it did not capture every long SVG completely, but it kept training feasible across all model sizes and allowed consistent scaling experiments.


## 7. Part 5.2 — Architecture Choices

I trained **decoder-only Transformer language models** because the task was next-token prediction over SVG code. A decoder-only architecture was appropriate because generation required autoregressively predicting the next SVG token from previous tokens.

I used five model scales: **Tiny, Small, Medium, Large, and XL**. These matched the project’s suggested scale pattern: increasing `d_model`, number of layers, number of attention heads, and feedforward dimension. This made the scaling study meaningful because parameter count increased systematically from roughly 1M parameters to around 88M parameters in the standard parameterization.

I used a context length of **512 tokens**. This was shorter than the preprocessing maximum of 2048 tokens, but it was a practical training choice. A 512-token context allowed training all five model sizes, including Large and XL, under the available Colab Pro GPU budget. Longer contexts would have increased attention cost quadratically and made the larger models substantially slower. Since many simplified icon SVGs were relatively short, 512 tokens still covered a useful portion of the dataset.

The feedforward dimension was set to approximately **4× the model dimension**, following common Transformer design practice. The number of heads was chosen so that each model dimension divided cleanly across heads. This made the architectures stable and easy to compare.

In [9]:
section_text = '\n## 7. Part 5.2 — Architecture Choices\n\nI trained **decoder-only Transformer language models** because the task was next-token prediction over SVG code. A decoder-only architecture was appropriate because generation required autoregressively predicting the next SVG token from previous tokens.\n\nI used five model scales: **Tiny, Small, Medium, Large, and XL**. These matched the project’s suggested scale pattern: increasing `d_model`, number of layers, number of attention heads, and feedforward dimension. This made the scaling study meaningful because parameter count increased systematically from roughly 1M parameters to around 88M parameters in the standard parameterization.\n\nI used a context length of **512 tokens**. This was shorter than the preprocessing maximum of 2048 tokens, but it was a practical training choice. A 512-token context allowed training all five model sizes, including Large and XL, under the available Colab Pro GPU budget. Longer contexts would have increased attention cost quadratically and made the larger models substantially slower. Since many simplified icon SVGs were relatively short, 512 tokens still covered a useful portion of the dataset.\n\nThe feedforward dimension was set to approximately **4× the model dimension**, following common Transformer design practice. The number of heads was chosen so that each model dimension divided cleanly across heads. This made the architectures stable and easy to compare.\n'
save_markdown(section_text, REPORT_TEXT_DIR / "part5_2_architecture_choices.md")
display(Markdown(section_text))

Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_2_architecture_choices.md



## 7. Part 5.2 — Architecture Choices

I trained **decoder-only Transformer language models** because the task was next-token prediction over SVG code. A decoder-only architecture was appropriate because generation required autoregressively predicting the next SVG token from previous tokens.

I used five model scales: **Tiny, Small, Medium, Large, and XL**. These matched the project’s suggested scale pattern: increasing `d_model`, number of layers, number of attention heads, and feedforward dimension. This made the scaling study meaningful because parameter count increased systematically from roughly 1M parameters to around 88M parameters in the standard parameterization.

I used a context length of **512 tokens**. This was shorter than the preprocessing maximum of 2048 tokens, but it was a practical training choice. A 512-token context allowed training all five model sizes, including Large and XL, under the available Colab Pro GPU budget. Longer contexts would have increased attention cost quadratically and made the larger models substantially slower. Since many simplified icon SVGs were relatively short, 512 tokens still covered a useful portion of the dataset.

The feedforward dimension was set to approximately **4× the model dimension**, following common Transformer design practice. The number of heads was chosen so that each model dimension divided cleanly across heads. This made the architectures stable and easy to compare.


## 8. Part 5.3 — Training Decisions

I used **AdamW** because it is a standard optimizer for Transformer language models and was recommended in the project. I used weight decay to provide mild regularization and gradient clipping to avoid unstable updates. Dropout was kept at zero in the main scaling runs to reduce confounding factors and make the scaling comparison cleaner.

I used a **cosine learning-rate schedule with warmup** for both standard and µP experiments. Warmup helped avoid unstable early updates, while cosine decay gradually reduced the learning rate during the epoch. This matched the project requirement and made the experiments comparable across models.

I kept the effective batch size constant in terms of tokens. With a micro-batch size of 16, gradient accumulation of 4, and a block size of 512, each optimizer step saw:

`16 × 4 × 512 = 32768 tokens`

Keeping the token batch size constant across model sizes ensured that validation losses were compared fairly.

I trained each model for exactly **one epoch** in Parts 2 and 3 for the scaling comparison. This ensured that all model sizes saw the same amount of data. For Part 4, I continued training the selected best model for an additional 2500 optimizer steps, corresponding to approximately 81.9M additional tokens, because the goal shifted from fair scaling comparison to improving the final generation model.

In [10]:
section_text = '\n## 8. Part 5.3 — Training Decisions\n\nI used **AdamW** because it is a standard optimizer for Transformer language models and was recommended in the project. I used weight decay to provide mild regularization and gradient clipping to avoid unstable updates. Dropout was kept at zero in the main scaling runs to reduce confounding factors and make the scaling comparison cleaner.\n\nI used a **cosine learning-rate schedule with warmup** for both standard and µP experiments. Warmup helped avoid unstable early updates, while cosine decay gradually reduced the learning rate during the epoch. This matched the project requirement and made the experiments comparable across models.\n\nI kept the effective batch size constant in terms of tokens. With a micro-batch size of 16, gradient accumulation of 4, and a block size of 512, each optimizer step saw:\n\n`16 × 4 × 512 = 32768 tokens`\n\nKeeping the token batch size constant across model sizes ensured that validation losses were compared fairly.\n\nI trained each model for exactly **one epoch** in Parts 2 and 3 for the scaling comparison. This ensured that all model sizes saw the same amount of data. For Part 4, I continued training the selected best model for an additional 2500 optimizer steps, corresponding to approximately 81.9M additional tokens, because the goal shifted from fair scaling comparison to improving the final generation model.\n'
save_markdown(section_text, REPORT_TEXT_DIR / "part5_3_training_decisions.md")
display(Markdown(section_text))

Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_3_training_decisions.md



## 8. Part 5.3 — Training Decisions

I used **AdamW** because it is a standard optimizer for Transformer language models and was recommended in the project. I used weight decay to provide mild regularization and gradient clipping to avoid unstable updates. Dropout was kept at zero in the main scaling runs to reduce confounding factors and make the scaling comparison cleaner.

I used a **cosine learning-rate schedule with warmup** for both standard and µP experiments. Warmup helped avoid unstable early updates, while cosine decay gradually reduced the learning rate during the epoch. This matched the project requirement and made the experiments comparable across models.

I kept the effective batch size constant in terms of tokens. With a micro-batch size of 16, gradient accumulation of 4, and a block size of 512, each optimizer step saw:

`16 × 4 × 512 = 32768 tokens`

Keeping the token batch size constant across model sizes ensured that validation losses were compared fairly.

I trained each model for exactly **one epoch** in Parts 2 and 3 for the scaling comparison. This ensured that all model sizes saw the same amount of data. For Part 4, I continued training the selected best model for an additional 2500 optimizer steps, corresponding to approximately 81.9M additional tokens, because the goal shifted from fair scaling comparison to improving the final generation model.


## 9. Part 5.4 — Scaling Insights

The project fit a power law of the form:

`L = aN^(-alpha) + c`

where `N` was parameter count and `L` was validation loss after one epoch.

The standard-parameterization curve did not produce a meaningful scaling exponent because the XL model became unstable and had much worse validation loss than the smaller models. Up to Large, the standard models followed the expected trend: increasing model size reduced validation loss. However, the XL point broke the monotonic trend, which caused the standard scaling-law fit to have an almost zero exponent and near-zero explanatory power.

The µP curve produced a much more meaningful scaling law. The µP fit had an exponent around **0.61** and an R² around **0.81** in my experiment. This was much steeper than commonly reported natural-language scaling exponents from Kaplan-style studies, where loss typically decreases more slowly with model size. However, I interpreted this cautiously because I only had five model sizes, one epoch per model, and a relatively small SVG-specific dataset compared with large natural-language corpora.

The results suggested that SVG code has a different structure from natural language. SVG contains strict syntax, repeated XML fragments, common icon templates, and many local patterns. These repeated structures may allow early improvements with scale to appear strong. At the same time, geometric coherence and controlled generation remained difficult, so lower validation loss did not automatically imply high-quality visual generation.

In [11]:
standard_alpha = safe(standard_fit, "alpha", "N/A")
standard_r2 = safe(standard_fit, "r2", "N/A")
mup_alpha = safe(mup_fit, "alpha", "N/A")
mup_r2 = safe(mup_fit, "r2", "N/A")

scaling_analysis = f'''## Scaling Insights

The project fit a power law of the form L = aN^(-alpha) + c, where N was the number of parameters and L was validation loss after one epoch.

The standard-parameterization curve did not produce a meaningful scaling exponent because the XL model became unstable. Up to the Large model, validation loss decreased with model size, but the XL model had much worse validation loss. This broke the monotonic scaling trend and made the standard power-law fit weak.

The standard fit had:
- alpha: {standard_alpha}
- R^2: {standard_r2}

The µP curve produced a more meaningful scaling fit because the large models remained more stable. The µP fit had:
- alpha: {mup_alpha}
- R^2: {mup_r2}

Compared with classic natural-language scaling-law work such as Kaplan et al. and Hoffmann et al., the SVG scaling exponent in this experiment should be interpreted cautiously. The fitted µP exponent was relatively steep, but the experiment used only five model sizes and one epoch per model. SVG code also had repeated XML syntax and icon templates, which may have made early scaling improvements sharper than in natural language. However, the generation results showed that better validation loss did not automatically mean reliable visual or prefix-conditioned generation.
'''
save_markdown(scaling_analysis, REPORT_TEXT_DIR / "part5_4_scaling_insights.md")
display(Markdown(scaling_analysis))

Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_4_scaling_insights.md


## Scaling Insights

The project fit a power law of the form L = aN^(-alpha) + c, where N was the number of parameters and L was validation loss after one epoch.

The standard-parameterization curve did not produce a meaningful scaling exponent because the XL model became unstable. Up to the Large model, validation loss decreased with model size, but the XL model had much worse validation loss. This broke the monotonic scaling trend and made the standard power-law fit weak.

The standard fit had:
- alpha: 5.628829028103905e-08
- R^2: -2.220446049250313e-16

The µP curve produced a more meaningful scaling fit because the large models remained more stable. The µP fit had:
- alpha: 0.6116495538335528
- R^2: 0.8116367896360249

Compared with classic natural-language scaling-law work such as Kaplan et al. and Hoffmann et al., the SVG scaling exponent in this experiment should be interpreted cautiously. The fitted µP exponent was relatively steep, but the experiment used only five model sizes and one epoch per model. SVG code also had repeated XML syntax and icon templates, which may have made early scaling improvements sharper than in natural language. However, the generation results showed that better validation loss did not automatically mean reliable visual or prefix-conditioned generation.


## 10. Part 5.5 — Learning-Rate Scaling Insights

In the standard-parameterization experiment, I selected the best learning rate from the Tiny model sweep and transferred it directly to all larger standard models. This worked well through the Large model, but it failed at XL. The standard XL model had a much worse validation loss than the smaller standard models, indicating that the Tiny-selected learning rate was too aggressive or otherwise poorly scaled for the largest standard model.

The µP experiment was designed to address exactly this issue. I used the `mup` package, `MuReadout`, `MuAdamW`, `set_base_shapes`, and modified attention scaling. The best learning rate selected on the Tiny µP model transferred much more reliably to larger µP models. The µP XL model did not collapse the way the standard XL model did.

The difference became most significant at the **XL scale**. Up to Large, standard parameterization had slightly better validation loss than µP in this run, but at XL the standard run degraded severely while µP remained stable. This showed that hyperparameter transfer can appear to work at moderate scales but fail abruptly when width/depth becomes large enough. µP helped by making learning-rate transfer more stable across model widths.

In [12]:
if standard_scaling is not None and mup_scaling is not None:
    lr_transfer_table = standard_scaling[["model_name", "params", "final_val_loss"]].rename(
        columns={"params": "standard_params", "final_val_loss": "standard_final_val_loss"}
    ).merge(
        mup_scaling[["model_name", "params", "final_val_loss"]].rename(
            columns={"params": "mup_params", "final_val_loss": "mup_final_val_loss"}
        ),
        on="model_name",
        how="inner"
    )
    lr_transfer_table["standard_minus_mup_val_loss"] = (
        lr_transfer_table["standard_final_val_loss"] - lr_transfer_table["mup_final_val_loss"]
    )
else:
    lr_transfer_table = pd.DataFrame()

display(lr_transfer_table)
lr_transfer_table.to_csv(TABLES_DIR / "fixed_lr_vs_mup_transfer_summary.csv", index=False)

lr_scaling_analysis = '''## Learning-Rate Scaling Insights

In the standard-parameterization experiment, I tuned the learning rate on the Tiny model and reused that same learning rate for all larger standard models. This transfer worked through the Large model but failed at XL. The standard XL model had much worse validation loss than the smaller models, so the standard scaling curve became unreliable.

The µP experiment tested whether a learning rate tuned on a small proxy model could transfer more reliably to larger widths. I used the mup package with MuReadout, MuAdamW, set_base_shapes, and modified attention scaling. Under µP, the XL model remained stable and achieved a much better validation loss than the standard XL model.

The difference became most significant at the XL scale. Up to Large, standard parameterization performed slightly better in validation loss, but at XL the fixed standard learning rate failed while µP remained stable. This showed that hyperparameter transfer can appear successful at moderate scales and then break at larger scales. µP improved the robustness of learning-rate transfer.
'''
save_markdown(lr_scaling_analysis, REPORT_TEXT_DIR / "part5_5_learning_rate_scaling_insights.md")
display(Markdown(lr_scaling_analysis))

,model_name,standard_params,standard_final_val_loss,mup_params,mup_final_val_loss,standard_minus_mup_val_loss
0,Tiny,1383168,0.911614,1907456,0.952466,-0.040851
1,Small,3554304,0.801394,4340736,0.839726,-0.038332
2,Medium,12417024,0.773271,13989888,0.824152,-0.050881
3,Large,33884160,0.767520,35981312,0.795803,-0.028283
4,XL,88594944,2.684681,91740672,0.829749,1.854933


Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_5_learning_rate_scaling_insights.md


## Learning-Rate Scaling Insights

In the standard-parameterization experiment, I tuned the learning rate on the Tiny model and reused that same learning rate for all larger standard models. This transfer worked through the Large model but failed at XL. The standard XL model had much worse validation loss than the smaller models, so the standard scaling curve became unreliable.

The µP experiment tested whether a learning rate tuned on a small proxy model could transfer more reliably to larger widths. I used the mup package with MuReadout, MuAdamW, set_base_shapes, and modified attention scaling. Under µP, the XL model remained stable and achieved a much better validation loss than the standard XL model.

The difference became most significant at the XL scale. Up to Large, standard parameterization performed slightly better in validation loss, but at XL the fixed standard learning rate failed while µP remained stable. This showed that hyperparameter transfer can appear successful at moderate scales and then break at larger scales. µP improved the robustness of learning-rate transfer.


## 11. Part 5.6 — SVG-Specific Patterns

The generated samples showed that the model learned many **surface-level SVG conventions**. It frequently generated valid `<svg>` wrappers, `viewBox` attributes, `<path>` elements, `fill`, `stroke`, `stroke-width`, `stroke-linecap`, and `stroke-linejoin`. These are common patterns in icon-like SVG datasets, and the model reproduced them in unconditional generations.

The model also learned some visual regularities. Many curated unconditional samples looked like icon fragments or simplified vector shapes rather than random text. This suggested that the model learned local SVG syntax and some common visual motifs. However, the model still struggled with precise geometry. Some generated paths contained out-of-range coordinates, repeated degenerate path commands, or visually strange shapes. This indicated that the model’s understanding of SVG was stronger at the syntactic/template level than at the global geometric level.

The prefix-conditioned outputs revealed a sharper limitation. The model often terminated immediately after partial prompts or failed to close the SVG root tag. The open-path prefix was the most successful case: in several candidates, the model closed a triangle-like path using a final line segment and `z`. However, even those completions were repetitive and visually sparse. This suggested that controlled continuation from arbitrary partial SVG prompts was much harder than unconditional generation.

I did not observe a clean phase transition from syntax to full spatial coherence. Instead, the results suggested gradual improvement in validation loss with model size, followed by training instability for the standard XL model. Qualitatively, the final model learned basic syntax and some icon-like patterns, but reliable spatial coherence and prefix-conditioned control remained limited.

In [13]:
if candidate_metrics is not None:
    raw_svg_paths = [Path(p) for p in candidate_metrics["svg_path"].dropna().tolist()]
    tag_counts = {}
    for path in raw_svg_paths:
        if path.exists():
            text = path.read_text(encoding="utf-8", errors="ignore")
            for match in re.finditer(r"<\s*([a-zA-Z0-9:_-]+)", text):
                tag = match.group(1).split(":")[-1].lower()
                if not tag.startswith("/"):
                    tag_counts[tag] = tag_counts.get(tag, 0) + 1
    tag_summary = pd.DataFrame(
        [{"tag": k, "count": v} for k, v in sorted(tag_counts.items(), key=lambda x: -x[1])]
    )
else:
    tag_summary = pd.DataFrame()

display(tag_summary.head(20))
tag_summary.to_csv(TABLES_DIR / "generated_svg_tag_frequency_summary.csv", index=False)

svg_patterns_analysis = '''## SVG-Specific Patterns

The generated samples showed that the model learned many surface-level SVG conventions. It frequently generated SVG wrappers, viewBox attributes, path elements, fill and stroke attributes, stroke widths, and rounded linecap/linejoin patterns. These are common in icon-style SVG datasets.

The curated unconditional samples showed that the model learned some visual motifs. Several outputs rendered as icon-like fragments or simplified vector shapes. However, the model still made geometric mistakes, including out-of-range coordinates, repeated degenerate paths, and visually strange shapes. This indicated that the model learned SVG syntax and common templates more strongly than global geometric coherence.

Prefix-conditioned generation was much weaker. Many prefix completions terminated immediately or failed to close the SVG tag. The open-path prefix sometimes worked partially: the model occasionally closed the triangle-like path. However, these completions were repetitive and visually sparse.

I did not observe a sharp phase transition into reliable spatial coherence. The project showed gradual quantitative improvement with scale, but qualitative generation remained limited by syntax errors, geometry artifacts, and weak prefix control.
'''
save_markdown(svg_patterns_analysis, REPORT_TEXT_DIR / "part5_6_svg_specific_patterns.md")
display(Markdown(svg_patterns_analysis))

,tag,count
0,path,321
1,svg,160
2,circle,52
3,rect,48
4,g,20
5,defs,2
6,style,2
7,line,2
8,polygon,1


Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_6_svg_specific_patterns.md


## SVG-Specific Patterns

The generated samples showed that the model learned many surface-level SVG conventions. It frequently generated SVG wrappers, viewBox attributes, path elements, fill and stroke attributes, stroke widths, and rounded linecap/linejoin patterns. These are common in icon-style SVG datasets.

The curated unconditional samples showed that the model learned some visual motifs. Several outputs rendered as icon-like fragments or simplified vector shapes. However, the model still made geometric mistakes, including out-of-range coordinates, repeated degenerate paths, and visually strange shapes. This indicated that the model learned SVG syntax and common templates more strongly than global geometric coherence.

Prefix-conditioned generation was much weaker. Many prefix completions terminated immediately or failed to close the SVG tag. The open-path prefix sometimes worked partially: the model occasionally closed the triangle-like path. However, these completions were repetitive and visually sparse.

I did not observe a sharp phase transition into reliable spatial coherence. The project showed gradual quantitative improvement with scale, but qualitative generation remained limited by syntax errors, geometry artifacts, and weak prefix control.


## 12. Part 5.7 — Challenges Encountered and Future Work

The biggest challenge was that **valid SVG syntax did not guarantee high-quality rendered output**. Some samples were technically valid and renderable but visually poor, blank, mostly black, or geometrically incoherent. For this reason, I separated quantitative evaluation over all candidates from curated visual examples selected by an automatic report-ready filter.

The second major challenge was **prefix-conditioned generation**. Even after generating many candidates, most prefix completions were invalid or low-information. Many completions terminated immediately after the prompt, and others failed to close the SVG root. This indicated that the model learned unconditional SVG patterns better than controlled continuation from arbitrary partial XML contexts.

The standard XL model was another important challenge. The fixed learning rate selected from the Tiny model transferred through Large but failed at XL. This made the standard scaling curve unreliable and showed why hyperparameter transfer is a real issue in scaling experiments.

With more time and resources, I would improve the project in several ways:

1. Train the best model for more epochs and use more data.
2. Increase context length beyond 512 tokens for longer SVGs.
3. Add constrained decoding or XML-aware generation to enforce closing tags and valid nesting.
4. Add grammar-based or parser-in-the-loop sampling to improve validity.
5. Train with augmentation or special formatting that separates tags, attributes, numbers, and path commands more explicitly.
6. Run a more complete hyperparameter sweep for larger standard models.
7. Train more model sizes to make the scaling-law fit more reliable.
8. Use visual similarity metrics or human evaluation, not only XML/render validity.
9. Explore prefix-conditioning training objectives or infilling-style objectives to improve controlled completion.

In [14]:
challenges_analysis = '''## Challenges and Future Work

The biggest challenge was that valid SVG syntax did not guarantee high-quality visual output. Some samples were technically valid and renderable but visually poor, blank, mostly black, or geometrically incoherent. For this reason, I reported quantitative metrics over all generated candidates and used an automatic visual filter only for selecting report figures.

The second major challenge was prefix-conditioned generation. Even after generating many candidates, most prefix completions were invalid or low-information. Many completions terminated immediately after the prompt or failed to close the SVG root. This showed that the model learned unconditional SVG patterns better than controlled continuation from arbitrary partial XML contexts.

Another challenge was the standard XL model. The Tiny-selected fixed learning rate transferred through Large but failed at XL. This made the standard scaling curve unreliable and showed why hyperparameter transfer matters in scaling experiments.

With more time and resources, I would train for more epochs, use more data, increase the context length, add constrained XML-aware decoding, run larger hyperparameter sweeps, train more model sizes, and add visual quality metrics beyond XML/render validity. I would also explore prefix-conditioning or infilling-specific training objectives to improve controlled SVG completion.
'''
save_markdown(challenges_analysis, REPORT_TEXT_DIR / "part5_7_challenges_future_work.md")
display(Markdown(challenges_analysis))

Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_7_challenges_future_work.md


## Challenges and Future Work

The biggest challenge was that valid SVG syntax did not guarantee high-quality visual output. Some samples were technically valid and renderable but visually poor, blank, mostly black, or geometrically incoherent. For this reason, I reported quantitative metrics over all generated candidates and used an automatic visual filter only for selecting report figures.

The second major challenge was prefix-conditioned generation. Even after generating many candidates, most prefix completions were invalid or low-information. Many completions terminated immediately after the prompt or failed to close the SVG root. This showed that the model learned unconditional SVG patterns better than controlled continuation from arbitrary partial XML contexts.

Another challenge was the standard XL model. The Tiny-selected fixed learning rate transferred through Large but failed at XL. This made the standard scaling curve unreliable and showed why hyperparameter transfer matters in scaling experiments.

With more time and resources, I would train for more epochs, use more data, increase the context length, add constrained XML-aware decoding, run larger hyperparameter sweeps, train more model sizes, and add visual quality metrics beyond XML/render validity. I would also explore prefix-conditioning or infilling-specific training objectives to improve controlled SVG completion.


## 13. Full Part 5 Report-Ready Analysis

In [15]:
full_part5_text = (
    (REPORT_TEXT_DIR / "part5_1_tokenization_strategy.md").read_text(encoding="utf-8") + "\n\n" +
    (REPORT_TEXT_DIR / "part5_2_architecture_choices.md").read_text(encoding="utf-8") + "\n\n" +
    (REPORT_TEXT_DIR / "part5_3_training_decisions.md").read_text(encoding="utf-8") + "\n\n" +
    (REPORT_TEXT_DIR / "part5_4_scaling_insights.md").read_text(encoding="utf-8") + "\n\n" +
    (REPORT_TEXT_DIR / "part5_5_learning_rate_scaling_insights.md").read_text(encoding="utf-8") + "\n\n" +
    (REPORT_TEXT_DIR / "part5_6_svg_specific_patterns.md").read_text(encoding="utf-8") + "\n\n" +
    (REPORT_TEXT_DIR / "part5_7_challenges_future_work.md").read_text(encoding="utf-8")
)

save_markdown(full_part5_text, REPORT_TEXT_DIR / "part5_full_design_decisions_analysis.md")
display(Markdown(full_part5_text[:5000]))

Saved: /content/drive/MyDrive/svg_gpt_scaling/final_report_assets/part5_design_analysis/report_text/part5_full_design_decisions_analysis.md



## 6. Part 5.1 — Tokenization Strategy

I used a **BPE tokenizer trained directly on the cleaned SVG corpus** rather than a natural-language tokenizer. This was appropriate because SVG code contains XML tags, numeric coordinates, path commands, attribute names, color values, and repeated syntax patterns that are very different from ordinary English. A domain-specific tokenizer allowed frequently occurring SVG fragments such as tags, attributes, and path substrings to become compact tokens instead of being split into inefficient character-level pieces.

I used a vocabulary size of **4096**, which was in the project-recommended range of 1K–8K. I chose 4096 as a middle-ground value: it was large enough to compress common SVG patterns, but small enough to keep the output softmax manageable for small models. A much smaller vocabulary would have increased sequence lengths and made the context window less effective. A much larger vocabulary would have increased embedding/output-layer parameters and could have been inefficient for the Tiny and Small models.

The preprocessing pipeline normalized SVG text before tokenization. I removed comments and metadata, normalized whitespace, rounded numeric precision, filtered very short and very long SVGs, validated XML, and deduplicated examples. This reduced vocabulary noise and made the token distribution more learnable.

The tokenizer produced a median SVG length of a few hundred tokens, while long examples could still exceed the context limit. For this reason, I filtered examples above **2048 tokens** and trained the Transformer using a context window of **512 tokens**. This was a practical compromise: it did not capture every long SVG completely, but it kept training feasible across all model sizes and allowed consistent scaling experiments.



## 7. Part 5.2 — Architecture Choices

I trained **decoder-only Transformer language models** because the task was next-token prediction over SVG code. A decoder-only architecture was appropriate because generation required autoregressively predicting the next SVG token from previous tokens.

I used five model scales: **Tiny, Small, Medium, Large, and XL**. These matched the project’s suggested scale pattern: increasing `d_model`, number of layers, number of attention heads, and feedforward dimension. This made the scaling study meaningful because parameter count increased systematically from roughly 1M parameters to around 88M parameters in the standard parameterization.

I used a context length of **512 tokens**. This was shorter than the preprocessing maximum of 2048 tokens, but it was a practical training choice. A 512-token context allowed training all five model sizes, including Large and XL, under the available Colab Pro GPU budget. Longer contexts would have increased attention cost quadratically and made the larger models substantially slower. Since many simplified icon SVGs were relatively short, 512 tokens still covered a useful portion of the dataset.

The feedforward dimension was set to approximately **4× the model dimension**, following common Transformer design practice. The number of heads was chosen so that each model dimension divided cleanly across heads. This made the architectures stable and easy to compare.



## 8. Part 5.3 — Training Decisions

I used **AdamW** because it is a standard optimizer for Transformer language models and was recommended in the project. I used weight decay to provide mild regularization and gradient clipping to avoid unstable updates. Dropout was kept at zero in the main scaling runs to reduce confounding factors and make the scaling comparison cleaner.

I used a **cosine learning-rate schedule with warmup** for both standard and µP experiments. Warmup helped avoid unstable early updates, while cosine decay gradually reduced the learning rate during the epoch. This matched the project requirement and made the experiments comparable across models.

I kept the effective batch size constant in terms of tokens. With a micro-batch size of 16, gradient accumulation of 4, and a block size of 512, each optimizer step saw:

`16 × 4 × 512 = 32768 tokens`

Keeping the token batch size constant across model sizes ensured that validation losses were compared fairly.

I trained each model for exactly **one epoch** in Parts 2 and 3 for the scaling comparison. This ensured that all model sizes saw the same amount of data. For Part 4, I continued training the selected best model for an additional 2500 optimizer steps, corresponding to approximately 81.9M additional tokens, because the goal shifted from fair scaling comparison to improving the final generation model.


## Scaling Insights

The project fit a power law of the form L = aN^(-alpha) + c, where N was the number of parameters and L was validation loss after one epoch.

The standard-parameterization curve did not produce a meaningful scaling exponent because the XL model became unstable. Up to the Large model, validation loss decreased w

## 14. Design Decision Summary Table and Key Figures

In [16]:
figures_to_copy = {
    "standard_scaling_law_real.png": NOTEBOOK2_DIR / "plots" / "standard_scaling_law_real.png",
    "standard_vs_mup_scaling_real.png": NOTEBOOK3_DIR / "plots" / "standard_vs_mup_scaling_real.png",
    "standard_vs_mup_lr_sweep_real.png": NOTEBOOK3_DIR / "plots" / "standard_vs_mup_lr_sweep_real.png",
    "scaling_extrapolation_10x_real.png": NOTEBOOK3_DIR / "plots" / "scaling_extrapolation_10x_real.png",
    "unconditional_generated_grid_curated.png": NOTEBOOK4_DIR / "report_assets" / "unconditional_generated_grid_curated.png",
    "all_generated_samples_grid_curated.png": NOTEBOOK4_DIR / "report_assets" / "all_generated_samples_grid_curated.png",
}

figure_rows = []
for dst_name, src in figures_to_copy.items():
    dst = FIGURES_DIR / dst_name
    copied = copy_if_exists(src, dst)
    figure_rows.append({"figure": dst_name, "source": str(src), "copied": copied, "final_path": str(dst)})

figure_manifest = pd.DataFrame(figure_rows)
display(figure_manifest)
figure_manifest.to_csv(TABLES_DIR / "part5_figure_manifest.csv", index=False)

design_decisions = pd.DataFrame([
    {"category": "Tokenization", "decision": "BPE tokenizer trained on cleaned SVG corpus", "justification": "SVG syntax differed from natural language; domain BPE captured repeated XML/path fragments."},
    {"category": "Vocabulary", "decision": "4096 tokens", "justification": "Middle of recommended 1K-8K range; balanced compression and model size."},
    {"category": "Context length", "decision": "512 tokens", "justification": "Kept attention cost manageable across five model sizes while covering many simplified icons."},
    {"category": "Model family", "decision": "Decoder-only Transformers from Tiny to XL", "justification": "Matched autoregressive SVG generation and enabled scaling-law fitting."},
    {"category": "Optimizer", "decision": "AdamW with cosine schedule and warmup", "justification": "Standard Transformer training setup and consistent comparison."},
    {"category": "Batch size", "decision": "32768 tokens per optimizer step", "justification": "Kept token batch size constant across model sizes."},
    {"category": "µP", "decision": "Used mup package with MuReadout, MuAdamW, set_base_shapes, attention scaling change", "justification": "Tested whether small-model LR transferred better to larger widths."},
    {"category": "Generation", "decision": "Generated many candidates and reported metrics over all candidates", "justification": "Separated honest quantitative evaluation from curated qualitative figures."},
])

display(design_decisions)
design_decisions.to_csv(TABLES_DIR / "part5_design_decisions_summary.csv", index=False)

print("Saved Part 5 tables and copied figures.")

,figure,source,copied,final_path
0,standard_scaling_law_real.png,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True,/content/drive/MyDrive/svg_gpt_scaling/final_r...
1,standard_vs_mup_scaling_real.png,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True,/content/drive/MyDrive/svg_gpt_scaling/final_r...
2,standard_vs_mup_lr_sweep_real.png,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True,/content/drive/MyDrive/svg_gpt_scaling/final_r...
3,scaling_extrapolation_10x_real.png,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True,/content/drive/MyDrive/svg_gpt_scaling/final_r...
4,unconditional_generated_grid_curated.png,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True,/content/drive/MyDrive/svg_gpt_scaling/final_r...
5,all_generated_samples_grid_curated.png,/content/drive/MyDrive/svg_gpt_scaling/noteboo...,True,/content/drive/MyDrive/svg_gpt_scaling/final_r...


,category,decision,justification
0,Tokenization,BPE tokenizer trained on cleaned SVG corpus,SVG syntax differed from natural language; dom...
1,Vocabulary,4096 tokens,Middle of recommended 1K-8K range; balanced co...
2,Context length,512 tokens,Kept attention cost manageable across five mod...
3,Model family,Decoder-only Transformers from Tiny to XL,Matched autoregressive SVG generation and enab...
4,Optimizer,AdamW with cosine schedule and warmup,Standard Transformer training setup and consis...
5,Batch size,32768 tokens per optimizer step,Kept token batch size constant across model si...
6,µP,"Used mup package with MuReadout, MuAdamW, set_...",Tested whether small-model LR transferred bett...
7,Generation,Generated many candidates and reported metrics...,Separated honest quantitative evaluation from ...


Saved Part 5 tables and copied figures.
